# Tiny GPT: Tokenizer First, Then 5D-Ready Parallel Training

This notebook now follows the full beginner pipeline in the right order:

1. install the required libraries
2. download TinyStories from Hugging Face
3. train a custom byte-level BPE tokenizer
4. tokenize the corpus into integer ids
5. build a small GPT-style Transformer
6. train the model with a project-aligned GPU parallelism scaffold
7. generate text

The design stays simple on purpose so you can see the complete workflow in one place.
The parallel path is also practical for Colab: it collapses to single-GPU when needed, uses `nn.DataParallel` inside the notebook when multiple GPUs are visible, and can switch to the repo's custom DDP wrappers plus sharded optimizer when launched under `torch.distributed`.


In [ ]:
# Run this cell first in Colab to install the required notebook packages.

import importlib
import subprocess
import sys


def ensure_package(module_name: str, pip_name: str | None = None) -> None:
    try:
        importlib.import_module(module_name)
        print(f"{module_name} is already installed.")
    except ImportError:
        package_name = pip_name or module_name
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])


ensure_package("torch")
ensure_package("datasets")
ensure_package("regex")
ensure_package("tqdm")

In [ ]:
import math
import os
import random
import socket
import sys
import warnings
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F

try:
    import regex as re
except ImportError:
    import re

try:
    from datasets import load_dataset
except ImportError:
    load_dataset = None


def find_repo_root() -> Path | None:
    """Walk upward from the notebook cwd until the repository root is found."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "GPT").exists():
            return candidate
    return None


REPO_ROOT = find_repo_root()
if REPO_ROOT is not None and str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

PROJECT_PARALLELISM_AVAILABLE = False
BucketedDistributedDataParallel = None
FlatDistributedDataParallel = None
IndividualParameterDistributedDataParallel = None
ProjectAdamW = None
ShardedOptimizer = None

if REPO_ROOT is not None:
    try:
        from GPT import (
            BucketedDistributedDataParallel,
            FlatDistributedDataParallel,
            IndividualParameterDistributedDataParallel,
            AdamW as ProjectAdamW,
            ShardedOptimizer,
        )
        PROJECT_PARALLELISM_AVAILABLE = True
    except Exception as exc:
        print("Project parallel wrappers unavailable:", type(exc).__name__, exc)

torch.manual_seed(42)
random.seed(42)


def resolve_device() -> tuple[torch.device, str]:
    """Pick the best available training device in the order CUDA, MPS, CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda"), "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps"), "mps"
    return torch.device("cpu"), "cpu"


def infer_primary_device(module: nn.Module | torch.Tensor | None = None) -> torch.device:
    """Infer the primary device for a module or tensor, defaulting to the global notebook device."""
    if isinstance(module, torch.Tensor):
        return module.device
    if isinstance(module, nn.Module):
        first_param = next(module.parameters(), None)
        if first_param is not None:
            return first_param.device
    return device


device, device_name = resolve_device()
cuda_device_count = torch.cuda.device_count() if torch.cuda.is_available() else 0

print(torch.__version__)
print("device:", device)
print("device type:", device_name)
print("visible cuda devices:", cuda_device_count)
print("repo root:", REPO_ROOT if REPO_ROOT is not None else "not found")
print("project parallel utilities:", PROJECT_PARALLELISM_AVAILABLE)


## Step 1: Load TinyStories

We train the tokenizer on **TinyStories** from Hugging Face.

To keep the notebook beginner-friendly and fast, we only use a small slice of the training split. You can increase the number of stories later if you want a better tokenizer.

In [ ]:
DATASET_NAME = "roneneldan/TinyStories"
MAX_STORIES = 500
END_OF_TEXT = "<|endoftext|>"


def load_tinystories_text(max_stories: int) -> tuple[str, list[str], str]:
    """Load TinyStories text, or fall back to a small local sample if offline."""
    if load_dataset is None:
        source = "offline fallback sample"
        stories = [
            "Once upon a time there was a small blue bird who loved to sing.",
            "A child found a shiny key under a tree and wondered what it could open.",
            "The moon looked bright over the quiet town while the wind moved softly.",
        ] * 200
        joined_text = END_OF_TEXT.join(stories)
        return joined_text, stories, source

    try:
        dataset = load_dataset(DATASET_NAME, split=f"train[:{max_stories}]")
        stories = [row["text"] for row in dataset if row["text"].strip()]
        source = f"{DATASET_NAME} train[:{max_stories}]"
    except Exception as exc:
        print("TinyStories download failed, using fallback sample instead.")
        print(type(exc).__name__, exc)
        source = "offline fallback sample"
        stories = [
            "Once upon a time there was a small blue bird who loved to sing.",
            "A child found a shiny key under a tree and wondered what it could open.",
            "The moon looked bright over the quiet town while the wind moved softly.",
        ] * 200

    joined_text = END_OF_TEXT.join(stories)
    return joined_text, stories, source


raw_text, stories, data_source = load_tinystories_text(MAX_STORIES)
print("data source:", data_source)
print("stories:", len(stories))
print("characters:", len(raw_text))
print(raw_text[:400])

## Step 2: Train a Custom Tokenizer

This notebook uses a simple **byte-level BPE tokenizer**.

Important ideas:
- every text string becomes UTF-8 bytes
- the base vocabulary starts with 256 single-byte tokens
- we repeatedly merge the most common adjacent byte pair
- the learned merge list becomes the tokenizer

This is a beginner version of the same general idea used in the project tokenizer code.

In [ ]:
PRETOKEN_PATTERN = re.compile(r"'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+")


def pretoken_to_bytes(pretoken: str) -> tuple[bytes, ...]:
    """Convert one pretoken into a tuple of one-byte tokens."""
    return tuple(bytes([b]) for b in pretoken.encode("utf-8"))


def iter_pairs(tokens: tuple[bytes, ...]):
    """Yield each adjacent pair in a token sequence."""
    return zip(tokens, tokens[1:])


def merge_pair_in_sequence(
    tokens: tuple[bytes, ...],
    pair: tuple[bytes, bytes],
    merged_token: bytes,
) -> tuple[bytes, ...]:
    """Merge one pair left to right inside a token sequence."""
    merged = []
    i = 0
    while i < len(tokens):
        if i + 1 < len(tokens) and tokens[i] == pair[0] and tokens[i + 1] == pair[1]:
            merged.append(merged_token)
            i += 2
        else:
            merged.append(tokens[i])
            i += 1
    return tuple(merged)


def split_with_special_tokens(text: str, special_tokens: list[str]) -> list[tuple[bool, str]]:
    """Split text into ordinary chunks and exact special-token matches."""
    if not special_tokens:
        return [(False, text)]

    sorted_tokens = sorted(special_tokens, key=len, reverse=True)
    pattern = re.compile("|".join(re.escape(token) for token in sorted_tokens))
    pieces = []
    cursor = 0

    for match in pattern.finditer(text):
        if match.start() > cursor:
            pieces.append((False, text[cursor:match.start()]))
        pieces.append((True, match.group(0)))
        cursor = match.end()

    if cursor < len(text):
        pieces.append((False, text[cursor:]))

    return pieces


class SimpleBPETokenizer:
    """A small byte-level BPE tokenizer for learning and experiments."""

    def __init__(
        self,
        vocab: dict[int, bytes],
        merges: list[tuple[bytes, bytes]],
        special_tokens: list[str] | None = None,
    ) -> None:
        self.vocab = dict(vocab)
        self.merges = list(merges)
        self.special_tokens = list(special_tokens or [])

        self.merge_ranks = {pair: rank for rank, pair in enumerate(self.merges)}
        self.id_to_bytes = dict(self.vocab)
        self.bytes_to_id = {token_bytes: token_id for token_id, token_bytes in self.id_to_bytes.items()}
        self._encode_cache = {}

    @classmethod
    def train(
        cls,
        text: str,
        vocab_size: int,
        special_tokens: list[str] | None = None,
    ) -> "SimpleBPETokenizer":
        """Train a small byte-level BPE tokenizer directly from text."""
        special_tokens = list(special_tokens or [])
        base_vocab_size = 256 + len(special_tokens)
        if vocab_size < base_vocab_size:
            raise ValueError(
                f"vocab_size must be at least {base_vocab_size} to include special tokens and bytes."
            )

        sequence_counts = Counter()
        for is_special, chunk in split_with_special_tokens(text, special_tokens):
            if is_special:
                continue
            for match in PRETOKEN_PATTERN.finditer(chunk):
                pretoken = match.group(0)
                if pretoken:
                    sequence_counts[pretoken_to_bytes(pretoken)] += 1

        vocab = {}
        next_id = 0
        for token in special_tokens:
            vocab[next_id] = token.encode("utf-8")
            next_id += 1

        for byte_value in range(256):
            vocab[next_id] = bytes([byte_value])
            next_id += 1

        vocab_values = set(vocab.values())
        merges = []

        while len(vocab) < vocab_size:
            pair_counts = Counter()
            for tokens, count in sequence_counts.items():
                for pair in iter_pairs(tokens):
                    pair_counts[pair] += count

            if not pair_counts:
                break

            best_pair = max(pair_counts.items(), key=lambda item: (item[1], item[0]))[0]
            merged_token = best_pair[0] + best_pair[1]
            merges.append(best_pair)

            if merged_token not in vocab_values:
                vocab[next_id] = merged_token
                vocab_values.add(merged_token)
                next_id += 1

            updated_counts = Counter()
            for tokens, count in sequence_counts.items():
                updated_tokens = merge_pair_in_sequence(tokens, best_pair, merged_token)
                updated_counts[updated_tokens] += count
            sequence_counts = updated_counts

        return cls(vocab=vocab, merges=merges, special_tokens=special_tokens)

    def _apply_merges(self, pretoken: str) -> list[bytes]:
        """Apply learned merges greedily by merge order."""
        tokens = list(pretoken_to_bytes(pretoken))
        while len(tokens) >= 2:
            best_rank = None
            best_pair = None

            for pair in iter_pairs(tuple(tokens)):
                rank = self.merge_ranks.get(pair)
                if rank is None:
                    continue
                if best_rank is None or rank < best_rank:
                    best_rank = rank
                    best_pair = pair

            if best_pair is None:
                break

            merged_token = best_pair[0] + best_pair[1]
            tokens = list(merge_pair_in_sequence(tuple(tokens), best_pair, merged_token))

        return tokens

    def encode(self, text: str) -> list[int]:
        """Encode text into tokenizer ids."""
        ids = []
        for is_special, chunk in split_with_special_tokens(text, self.special_tokens):
            if not chunk:
                continue
            if is_special:
                ids.append(self.bytes_to_id[chunk.encode("utf-8")])
                continue

            for match in PRETOKEN_PATTERN.finditer(chunk):
                pretoken = match.group(0)
                if pretoken in self._encode_cache:
                    ids.extend(self._encode_cache[pretoken])
                    continue

                merged_tokens = self._apply_merges(pretoken)
                token_ids = [self.bytes_to_id[token] for token in merged_tokens]
                self._encode_cache[pretoken] = token_ids
                ids.extend(token_ids)

        return ids

    def decode(self, ids: list[int]) -> str:
        """Decode tokenizer ids back into text."""
        raw_bytes = b"".join(self.id_to_bytes[token_id] for token_id in ids)
        return raw_bytes.decode("utf-8", errors="replace")

In [ ]:
TOKENIZER_VOCAB_SIZE = 384
SPECIAL_TOKENS = [END_OF_TEXT]

tokenizer = SimpleBPETokenizer.train(
    text=raw_text,
    vocab_size=TOKENIZER_VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS,
)

print("tokenizer vocab size:", len(tokenizer.vocab))
print("number of learned merges:", len(tokenizer.merges))

sample_text = stories[0][:200]
sample_ids = tokenizer.encode(sample_text)

print("sample text:")
print(sample_text)
print()
print("sample token ids:")
print(sample_ids[:80])
print()
print("decoded sample:")
print(tokenizer.decode(sample_ids))

## Step 3: Tokenize the Corpus

Now that the tokenizer is trained, we convert the full TinyStories text into integer token ids.

These ids are what the GPT model will see during training.

In [ ]:
all_token_ids = tokenizer.encode(raw_text)
token_data = torch.tensor(all_token_ids, dtype=torch.long)

split_index = int(0.9 * len(token_data))
train_data = token_data[:split_index]
val_data = token_data[split_index:]

print("total tokens:", len(token_data))
print("train tokens:", len(train_data))
print("val tokens:", len(val_data))
print("average characters per token:", round(len(raw_text) / len(token_data), 3))

## Step 4: Build Tiny GPT

Now that we already have token ids, we can build the GPT model.

This notebook keeps the Transformer simple:
- `LayerNorm`
- learned position embeddings
- causal multi-head self-attention
- a basic `GELU` feed-forward network

The project code contains more advanced versions, but the overall structure is the same.

In [ ]:
@dataclass
class TinyGPTConfig:
    """Simple configuration container for the model."""

    vocab_size: int
    block_size: int = 128
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    dropout: float = 0.1

    def __post_init__(self):
        if self.n_embd % self.n_head != 0:
            raise ValueError("n_embd must be divisible by n_head.")


class CausalSelfAttention(nn.Module):
    """Masked multi-head self-attention for a decoder-only Transformer."""

    def __init__(self, config: TinyGPTConfig):
        super().__init__()
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head

        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        mask = torch.tril(torch.ones(config.block_size, config.block_size))
        self.register_buffer(
            "causal_mask",
            mask.view(1, 1, config.block_size, config.block_size),
            persistent=False,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, channels = x.shape

        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

        q = q.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)

        attention_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attention_scores = attention_scores.masked_fill(
            self.causal_mask[:, :, :seq_len, :seq_len] == 0,
            float("-inf"),
        )

        attention_weights = F.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)

        y = attention_weights @ v
        y = y.transpose(1, 2).contiguous().view(batch_size, seq_len, channels)
        y = self.c_proj(y)
        return self.resid_dropout(y)


class FeedForward(nn.Module):
    """Simple MLP used after attention in each Transformer block."""

    def __init__(self, config: TinyGPTConfig):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TransformerBlock(nn.Module):
    """One full Transformer block: norm, attention, norm, and MLP."""

    def __init__(self, config: TinyGPTConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = FeedForward(config)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class TinyGPT(nn.Module):
    """A small GPT-style language model built from the blocks above."""

    def __init__(self, config: TinyGPTConfig):
        super().__init__()
        self.config = config

        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding = nn.Embedding(config.block_size, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layer)])
        self.final_norm = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        self.lm_head.weight = self.token_embedding.weight
        self.apply(self._init_weights)

    def _init_weights(self, module: nn.Module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        batch_size, seq_len = idx.shape
        if seq_len > self.config.block_size:
            raise ValueError(
                f"Sequence length {seq_len} is bigger than block size {self.config.block_size}."
            )

        positions = torch.arange(seq_len, device=idx.device)
        token_embeddings = self.token_embedding(idx)
        position_embeddings = self.position_embedding(positions)

        x = self.dropout(token_embeddings + position_embeddings)
        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(batch_size * seq_len, self.config.vocab_size),
                targets.reshape(batch_size * seq_len),
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int, temperature: float = 1.0):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.block_size :]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

## Step 5: Training Helpers

These helper functions follow the same language-model training idea used in the project:
- sample random token windows
- predict the next token
- evaluate train and validation loss over several batches
- map the notebook onto a 5-axis parallelism plan: data, tensor, pipeline, context, and sequence

In this tiny notebook, the executable runtime materializes the data-parallel axis directly and keeps the other axes as topology placeholders unless you extend the model further. That keeps the notebook runnable on Colab while still matching the project-style parallel configuration vocabulary.


In [ ]:
@dataclass
class FiveDParallelConfig:
    """A small topology container that mirrors a 5-axis parallel training plan."""

    data_parallel: int = 1
    tensor_parallel: int = 1
    pipeline_parallel: int = 1
    context_parallel: int = 1
    sequence_parallel: int = 1
    ddp_strategy: str = "bucketed"
    bucket_size_mb: float = 8.0
    use_sharded_optimizer: bool = True

    @property
    def requested_world_size(self) -> int:
        return (
            self.data_parallel
            * self.tensor_parallel
            * self.pipeline_parallel
            * self.context_parallel
            * self.sequence_parallel
        )

    def as_dict(self) -> dict[str, int | float | bool | str]:
        return {
            "dp": self.data_parallel,
            "tp": self.tensor_parallel,
            "pp": self.pipeline_parallel,
            "cp": self.context_parallel,
            "sp": self.sequence_parallel,
            "ddp_strategy": self.ddp_strategy,
            "bucket_size_mb": self.bucket_size_mb,
            "use_sharded_optimizer": self.use_sharded_optimizer,
        }


def normalize_parallel_config(config: FiveDParallelConfig) -> FiveDParallelConfig:
    """Collapse unsupported axes while preserving a 5D-style configuration surface."""
    values = [
        config.data_parallel,
        config.tensor_parallel,
        config.pipeline_parallel,
        config.context_parallel,
        config.sequence_parallel,
    ]
    if any(value <= 0 for value in values):
        raise ValueError("All parallelism dimensions must be positive integers.")

    if device_name != "cuda":
        return FiveDParallelConfig(
            data_parallel=1,
            tensor_parallel=1,
            pipeline_parallel=1,
            context_parallel=1,
            sequence_parallel=1,
            ddp_strategy=config.ddp_strategy,
            bucket_size_mb=config.bucket_size_mb,
            use_sharded_optimizer=False,
        )

    active = FiveDParallelConfig(**config.__dict__)
    unsupported_axes = []
    for axis_name in ("tensor_parallel", "pipeline_parallel", "context_parallel", "sequence_parallel"):
        axis_value = getattr(active, axis_name)
        if axis_value > 1:
            unsupported_axes.append(f"{axis_name}={axis_value}")
            setattr(active, axis_name, 1)

    if unsupported_axes:
        warnings.warn(
            "This notebook exposes a 5D topology config, but the in-notebook runtime only executes "
            "the data-parallel axis directly. Collapsing unsupported axes to 1: " + ", ".join(unsupported_axes),
            stacklevel=2,
        )

    active.data_parallel = max(1, min(active.data_parallel, cuda_device_count))
    if active.data_parallel != config.data_parallel:
        warnings.warn(
            f"Requested data_parallel={config.data_parallel}, but only {cuda_device_count} CUDA device(s) are visible. "
            f"Using data_parallel={active.data_parallel} instead.",
            stacklevel=2,
        )

    if dist.is_initialized() and dist.get_world_size() != active.data_parallel:
        warnings.warn(
            f"torch.distributed is initialized with world_size={dist.get_world_size()}, so the active data-parallel axis "
            f"is forced to {dist.get_world_size()}.",
            stacklevel=2,
        )
        active.data_parallel = dist.get_world_size()

    if not dist.is_initialized():
        active.use_sharded_optimizer = False

    return active


def summarize_parallel_plan(requested: FiveDParallelConfig, active: FiveDParallelConfig) -> None:
    """Print a compact summary of the requested and active topology."""
    print("requested 5D mesh:", requested.as_dict())
    print("active 5D mesh:", active.as_dict())
    print("requested world size:", requested.requested_world_size)
    print("active world size:", active.requested_world_size)
    if dist.is_initialized():
        print("runtime: torch.distributed")
    elif active.data_parallel > 1 and device_name == "cuda":
        print("runtime: nn.DataParallel")
    else:
        print("runtime: single process")


def unwrap_model(model: nn.Module) -> nn.Module:
    """Return the underlying TinyGPT module regardless of the wrapper type."""
    if hasattr(model, "module"):
        return model.module
    return model


def finalize_parallel_backward(model: nn.Module) -> None:
    """Finish custom DDP gradient synchronization when the wrapper exposes it."""
    if hasattr(model, "finish_gradient_synchronization"):
        model.finish_gradient_synchronization()


def training_device_for_runtime(active: FiveDParallelConfig) -> torch.device:
    """Return the device that batches should be moved to before the forward pass."""
    if device_name == "cuda" and active.data_parallel > 1:
        return torch.device("cuda:0")
    return device


def prepare_parallel_model(model: TinyGPT, active: FiveDParallelConfig) -> tuple[nn.Module, torch.device]:
    """Move the model onto the best available parallel wrapper for the notebook runtime."""
    train_device = training_device_for_runtime(active)
    model = model.to(train_device)

    if dist.is_initialized() and active.data_parallel > 1 and PROJECT_PARALLELISM_AVAILABLE:
        strategy = active.ddp_strategy.lower()
        if strategy == "flat":
            return FlatDistributedDataParallel(model), train_device
        if strategy == "individual":
            return IndividualParameterDistributedDataParallel(model), train_device
        return BucketedDistributedDataParallel(model, bucket_size_mb=active.bucket_size_mb), train_device

    if device_name == "cuda" and active.data_parallel > 1:
        device_ids = list(range(active.data_parallel))
        return nn.DataParallel(model, device_ids=device_ids), train_device

    return model, train_device


def build_optimizer(
    model: nn.Module,
    *,
    lr: float,
    weight_decay: float,
    active: FiveDParallelConfig,
) -> torch.optim.Optimizer:
    """Build the notebook optimizer, optionally using the project's sharded optimizer."""
    params = unwrap_model(model).parameters()
    if active.use_sharded_optimizer and PROJECT_PARALLELISM_AVAILABLE and dist.is_initialized():
        return ShardedOptimizer(
            params,
            ProjectAdamW,
            lr=lr,
            weight_decay=weight_decay,
        )
    return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)


def get_batch(
    source: torch.Tensor,
    batch_size: int,
    block_size: int,
    target_device: torch.device,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Sample random token windows and their next-token targets."""
    starts = torch.randint(0, len(source) - block_size - 1, (batch_size,))
    x = torch.stack([source[i : i + block_size] for i in starts])
    y = torch.stack([source[i + 1 : i + block_size + 1] for i in starts])
    return x.to(target_device), y.to(target_device)


def normalize_loss(loss: torch.Tensor | None) -> torch.Tensor | None:
    """Reduce wrapper-specific loss shapes to one scalar."""
    if loss is None:
        return None
    if loss.ndim == 0:
        return loss
    return loss.mean()


@torch.no_grad()
def estimate_loss(
    model: nn.Module,
    train_data: torch.Tensor,
    val_data: torch.Tensor,
    eval_iters: int,
    batch_size: int,
    target_device: torch.device,
) -> dict[str, float]:
    """Average train and validation loss over a few random batches."""
    losses = {}
    base_model = unwrap_model(model)
    model.eval()
    for split_name, split_data in [("train", train_data), ("val", val_data)]:
        split_losses = []
        for _ in range(eval_iters):
            x, y = get_batch(split_data, batch_size, base_model.config.block_size, target_device)
            _, loss = model(x, y)
            loss = normalize_loss(loss)
            assert loss is not None
            split_losses.append(float(loss.detach().cpu()))
        losses[split_name] = sum(split_losses) / len(split_losses)
    model.train()
    return losses


## Step 6: Train Tiny GPT

Now we train GPT on the tokenized TinyStories data.

This parallel notebook keeps the model small, but the runtime now accepts a 5-axis topology description:
- data parallel
- tensor parallel
- pipeline parallel
- context parallel
- sequence parallel

The notebook executes the data-parallel axis directly, uses the repo's custom DDP wrappers and sharded optimizer when `torch.distributed` is already active, and otherwise falls back to in-notebook multi-GPU `DataParallel` when multiple CUDA devices are visible.


In [ ]:
requested_parallel_config = FiveDParallelConfig(
    data_parallel=min(2, cuda_device_count) if device_name == "cuda" else 1,
    tensor_parallel=1,
    pipeline_parallel=1,
    context_parallel=1,
    sequence_parallel=1,
    ddp_strategy="bucketed",
    bucket_size_mb=8.0,
    use_sharded_optimizer=True,
)
active_parallel_config = normalize_parallel_config(requested_parallel_config)
summarize_parallel_plan(requested_parallel_config, active_parallel_config)

train_config = TinyGPTConfig(
    vocab_size=len(tokenizer.vocab),
    block_size=64,
    n_layer=2,
    n_head=4,
    n_embd=128,
    dropout=0.1,
)

batch_size = 16
max_iters = 60
eval_interval = 20
eval_iters = 5
grad_clip = 1.0
learning_rate = 3e-4
weight_decay = 0.01

if batch_size % active_parallel_config.data_parallel != 0:
    raise ValueError(
        "batch_size must be divisible by the active data-parallel degree so each replica gets the same number of samples."
    )

model, training_device = prepare_parallel_model(TinyGPT(train_config), active_parallel_config)
optimizer = build_optimizer(
    model,
    lr=learning_rate,
    weight_decay=weight_decay,
    active=active_parallel_config,
)

print("training device:", training_device)
print("model wrapper:", type(model).__name__)

for step in range(max_iters):
    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss(
            model,
            train_data,
            val_data,
            eval_iters,
            batch_size,
            training_device,
        )
        print(
            f"step {step:03d} | train loss {losses['train']:.4f} | val loss {losses['val']:.4f}"
        )

    x, y = get_batch(train_data, batch_size, train_config.block_size, training_device)
    _, loss = model(x, y)
    loss = normalize_loss(loss)
    assert loss is not None

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    finalize_parallel_backward(model)
    torch.nn.utils.clip_grad_norm_(unwrap_model(model).parameters(), grad_clip)
    optimizer.step()

print("final training loss:", float(loss.detach().cpu()))


## Step 7: Generate Text

We encode a prompt with the custom tokenizer, generate more token ids with GPT, and decode those ids back into text.

In [ ]:
prompt = "Once upon a time"
prompt_ids = tokenizer.encode(prompt)

base_model = unwrap_model(model)
base_model.eval()

context = torch.tensor([prompt_ids], dtype=torch.long, device=training_device)
generated_ids = base_model.generate(context, max_new_tokens=80)
generated_text = tokenizer.decode(generated_ids[0].detach().cpu().tolist())
print(generated_text)


## How This Connects to the Project

This notebook still follows the same end-to-end learning order:
- dataset
- tokenizer
- token ids
- GPT model
- training
- generation

The parallel notebook now also uses the project's parallel-training vocabulary and, when available, its runtime pieces:
- a 5-axis topology config: data, tensor, pipeline, context, and sequence parallelism
- the repo's custom DDP wrappers from `GPT/ddp.py`
- the repo's optimizer-state sharding wrapper from `GPT/optimization.py`

Main simplifications in this notebook:
- the tiny model is still written directly in the notebook so the teaching path stays compact
- only the data-parallel axis is executed directly inside the notebook runtime
- the remaining 5D axes are exposed as topology placeholders and collapse to `1` unless you extend the model further
- the training run stays short so the Colab demo finishes quickly
